# **Measure Organelle Interactions**

***Prior to this notebook, you should have already run through [2.0_quantification_setup](2.0_quantification_setup.ipynb).***

The `'methods_...'` notebooks included here in infer-subc Part 2: quantification will go over how each of the quantification methods (morphology, interactions, and distribution) are carried out. The notebooks will explain each step in the method and display the combined function at the end of the notebook. 

### 🍃 **Biological Relevance - Organelle Interactions**
Intracellular organelles do not exist independently of one another. In recent years, the existance and function of organelle contact site, regions of close apposition between membrane bound organelles has been recognized. They facilitate protein, lipid, and metabolite transport, coordinate organelle trafficking and function, and are involved in many cellular pathways and functions [[1](https://www.cell.com/cell/pdf/S0092-8674(23)01328-4.pdf)]. 

The distance between membranes at contact sites is between 30-80 nm depending on the types of organelles involved. This distance is not resolvable using standard confocal microscopy images [[2](https://zeiss-campus.magnet.fsu.edu/articles/basics/resolution.html)]. However, interactions between organelle which can include organelle contact sites can be estimated through overlap in label localization in confocal microscopy images.

### **Oragnelle Interactions** 📐
Here, we will use the overlapping area between two or more organelle objects as putative interaction sites. The current notebook is formatted to quantify all the n-way overlappings of one type between n organelles; as such, this notebook is not designed for measuring ALL the interactions from one cell instead, the [organelle_interactions](2.2_organelle_interactions.ipynb) notebook outlines how we analyze all types interactions at a time.

**`Organelle interaction sites`**: regions of overlap between two or more organelles

These sites can then be measured for features such as number, size, and shape utilizing the `get_morpholgy_metrics()` function outlined in [method_morphology](method_morphology.ipynb) notebook and/or measurements of subcellular distribution utilizing the `get_distribution()` function outlined in [method_distribution](method_distribution.ipynb) notebook.

-----

### 👣 **Summary of steps**  

🛠️ **BUILD INTERACTION FUNCTIONS STEP-BY-STEP**

- **`STEP 01`** - Select organelles of interest
- **`STEP 02`** - Loop through list of organelles involved in the interaction to add to overlap
- **`STEP 03`** - Apply cell mask to the overlap segmentation
- **`STEP 04`** - Run regionprops for the overlap segmentation
- **`STEP 05`** - Determine the cells and subregions the overlaps are located in
- **`STEP 06`** - Determine surface area of the overlaps
- **`STEP 07`** - Determine the organelles involved in the overlaps
- **`STEP 08`** - Add columns to tables for the cell number, subregion number, and organelles involved
- **`STEP 09`** - (OPTIONAL) Run distribution metrics on the interaction metrics
- **`STEP 10`** - Determine which overlaps are also present in higher order overlaps
- **`STEP 11`** - Add column to interaction table for the presence of the interaction in a higher order overlap

⚙️ **DEFINE AND TEST *`Interactions`* FUNCTIONS**
- Define `create_overlap` function
- Run `create_overlap` function

- Define `interactions_metric_analysis` function
- Run `interactions_metric_analysis` function

- Define `find_novel_overlap` function
- Run `find_novel_overlaps` function

The above steps will be applied to set of organelles of interest. Batch processing is available in a separate notebook.

---------------------
## **IMPORTS AND LOAD IMAGE**
Details about the functions included in this subsection are outlined in the [`2.0_quantification_setup`](2.0_quantification_setup.ipynb) notebook. Please visit that notebook first if you are confused about any of the code included here.

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
import os
from pathlib import Path
import napari
import numpy as np
import pandas as pd
import itertools
from typing import Union
from skimage.measure import regionprops_table, label
from infer_subc.core.img import apply_mask
from infer_subc.core.file_io import (read_czi_image,
                                     read_tiff_image,
                                     list_image_files)
from infer_subc.utils.batch import (list_image_files, 
                                    find_segmentation_tiff_files,
                                    cell_finder, 
                                    region_finder,
                                    make_dict)
from infer_subc.quantification.stats import surface_area_from_props
from infer_subc.quantification.distribution import (get_XY_distribution, 
                                                    get_Z_distribution)
from infer_subc.quantification.interactions import (create_overlap, 
                                                    interaction_metric_analysis,
                                                    find_novel_overlaps)
splitter = "X"

###################
## Color Constants
###################
ORANGE  = '#FFA500'
BOPBLUE = '#20ADF8'
MAROON  = '#800000'
WHITE   = '#FFFFFF'
BLACK   = '#000000'


#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: `raw_img_type`, `data_root_path`, `raw_data_path`, `seg_data_path`, and `quant_data_path`.

In [ ]:
#### USER INPUT REQUIRED ###
raw_img_type = ".czi"
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python Scripts/Infer-subc-2D"
raw_data_path = data_root_path / "raw_single"
seg_data_path = data_root_path / "out_single"
quant_data_path = data_root_path / "quant_single"
print(str(quant_data_path))

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
# Create the output directory to save the segmentation outputs in.
if not Path.exists(quant_data_path):
    Path.mkdir(quant_data_path)
    print(f"making {quant_data_path}")

# Create a list of the file paths for each image in the input folder. Select test image path.
raw_img_file_list = list_image_files(raw_data_path,raw_img_type)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.DataFrame({"Image Name":raw_img_file_list})

#### &#x1F6D1; &#x270D; **User Input Required:**

Use the list above to specify which image you wish to analyze based on its index: `test_img_n`

In [ ]:
#### USER INPUT REQUIRED ###
test_img_n = 0

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
# Read in the image and metadata as an ndarray and dictionary from the test image selected above. 
test_img_name = raw_img_file_list[test_img_n]
img_data,meta_dict = read_czi_image(test_img_name)

# Define some of the metadata features.
channel_names = meta_dict['name']
meta = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']
file_path = meta_dict['file_name']

print("Metadata information")
print(f"File path: {file_path}")
for i in list(range(len(channel_names))):
    print(f"Channel {i} name: {channel_names[i]}")
print(f"Scale (ZYX): {scale}")
print(f"Channel axis: {channel_axis}")

#### &#x1F6D1; &#x270D; **User Input Required:**

Specify the following information about the segmentation files: - `org_file_names`, `org_channels_ordered`, `regions_file_names`, `suffix_separator`, and `mask_name`.

In [ ]:
#### USER INPUT REQUIRED ###
org_file_names = ["lyso", "mito", "golgi", "perox", "ER", "LD"]
org_channels_ordered = [4,3,2,1,0,6]
regions_file_names = ["cell", "nuc", "soma", "neurites"]
subregions_file_names = ["soma", "neurites"]
suffix_separator = "-20230426_test_"
mask_name = "cell"
include_dist = True
dist_centering_obj = ['nuc', 'nuc', None]
dist_center_on=False
dist_keep_center_as_bin=True
dist_zernike_degrees=None
dist_num_bins = 5

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
# find file paths for segmentations
all_suffixes = org_file_names + regions_file_names
filez = find_segmentation_tiff_files(file_path, all_suffixes, seg_data_path, suffix_separator)

# read the segmentation and masks/regions files into memory
organelle_segs_list = [read_tiff_image(filez[org]) for org in org_file_names]
organelle_segs = make_dict(list_obj_names = org_file_names, list_obj_segs = organelle_segs_list)

regions = [] 
regions_dict = {}
for m in regions_file_names:
    mfile = read_tiff_image(filez[m])
    regions.append(mfile)
    regions_dict[m] = mfile
    if m == mask_name:
        mask = mfile

# match the intensity channels to the segmentation files
intensities = [img_data[ch] for ch in org_channels_ordered]

# open viewer and add images
viewer = napari.Viewer()
for r, reg in enumerate(regions_file_names):
    viewer.add_image(regions[r],
                     scale=scale,
                     name=f"{reg} mask")

# colors = ["red", "bop orange", "yellow", "green", "blue", "cyan", "magenta", "bop purple"]
for o, org in enumerate(org_file_names):
    viewer.add_image(intensities[o],
                     scale=scale,
                     name=f"{org} intensity channel")
    viewer.add_labels(organelle_segs[org],
                      scale=scale,
                      name=f"{org} segmentation")
viewer.grid.enabled = True
viewer.reset_view()

print("The following matching files were found and can now be viewed in Napari:")
filez

-----
## **QUANTIFY INTERACTIONS BETWEEN *n organelles* FROM <INS>ONE CELL</INS>**

### **`STEP 1` - Select organelles of interest**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The below block of code lists out the possible combinations of organelle overlaps and displays each of them assigned to a unique number. This unique number should be used to select the desired interaction group.

In [ ]:
all_pos =[]
for n in list(map(lambda x:x+2, (range(len(org_file_names)-1)))):
    all_pos += itertools.combinations(org_file_names, n)
possib = [splitter.join(inter) for inter in all_pos]

pd.DataFrame({"Overlapping Organelles": possib})

#### &#x1F6D1; &#x270D; **User Input Required:**

Choose desired combo number from above list.

In [ ]:
overlap_orgs = 17

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** Here, we print out the names of the organelles used in the chosen overlap for verification that the organelles chosen to make an overlap with are correct.

In [ ]:
overlap_orgs = possib[overlap_orgs]
print(overlap_orgs)

#### **`STEP 2`** - Loop through list of organelles involved in the interaction to add to overlap

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The segmentations of the organelles chosen above to use in overlaps are overlapped in the below step. This overlapping works for declumped segmentations by combining the values of two different segmentations, but retaining their values in different digits of the combined segmentation. The overlapped region segmentation is then set to only the values present where overlaps occur. Next, the remaining segmentation is relabeled to simplify the memory requirement from overlapping the organelles to make a new overlap segmentation. Finally, if there are more than two organelles present within the desired overlap, this process is repeated using the new overlap segmentation in the place of one of the two organelles until there are no more organelles desired to be overlapped.

In [ ]:
#### **`STEP 1`** - Loop through list of organelles involved in the interaction to add to overlap
site = np.ones_like(organelle_segs[overlap_orgs.split(splitter)[0]])
for org in overlap_orgs.split(splitter):
    b = organelle_segs[org]             
    valid = (b>0)*(site>0)
    digit = len(str(np.max(site)))      
    site = (b*(10**(digit)))+site       
    site[valid.astype(bool)==False]=0   
    site = label(site)          

#### **`STEP 3`** - Apply Cell Mask

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** The below code ensures that the overlap segmentation is isolated to only the area within the cell mask.

In [ ]:
site = apply_mask(site, mask)

#### **`STEP 4`** - Run Regionprops

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block runs the regionprops function. For more details on how the regionprops function works, please check the method_morphology.ipynb jupyter notebook.

In [ ]:
##########################################
## CREATE LIST OF REGIONPROPS MEASUREMENTS
##########################################
# start with LABEL
properties = ["label"]

# add position
properties += ["centroid", "bbox"]

# add area
properties += ["area", "equivalent_diameter"] # "num_pixels", 

# add shape measurements - NOTE: can't include minor axis measure because some of the contact sites are only one pixel
properties += ["extent", "euler_number", "solidity", "axis_major_length", "slice"] # "feret_diameter_max",  , "axis_minor_length"
    

##################
## RUN REGIONPROPS
##################
props = regionprops_table(site, 
                          intensity_image=None, 
                          properties=properties, 
                          extra_properties=None, 
                          spacing=scale)

#### **`STEP 5`** - Determine locations of overlaps

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** Here we find the locations of the overlaps--we determine the cell number and the region in which the overlap is located. If an organelle is found to not be present in any region or to not be present in any cell, an error will be thrown. The likely culprit of this happening is due to manual editing of the cell mask file leaving small objects that are not connected to the main cell.

In [ ]:
cells = cell_finder(scale=scale, obj=site, mask=mask, props=props)
subregions = region_finder(scale=scale, obj=site, regions=regions_dict, props=props)

#### **`STEP 6`** - Determine surface area of overlaps

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This block runs the function to measure the surface area of an object from the regionprops output. For more details on how this works, check the method_morphology.ipynb jupyter notebook.

In [ ]:
##################################################################
## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
##################################################################
surface_area_tab = pd.DataFrame(surface_area_from_props(site, props, scale))

#### **`STEP 7`** - Determine organelles involved in interactions by ID

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** We determine organelles involved in interactions by ID using the region_props table. To do so, we compare the interaction location with involved organelles and collect the data for the organelle ID for each individual organelle involved within each interaction site. These IDs are then combined into a single ID separated by '_'. This works similarly to the region_finder and cell_finder functions.

In [ ]:
########################################################
## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
########################################################
over_inv = []
involved = overlap_orgs.split(splitter)
indexes = {overlap_orgs: []}
for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = site[props["slice"][index]]
            lorg = organelle_segs[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}")
            over_inv.append(f"{all_inv[0]}")
        indexes[overlap_orgs].append('_'.join(over_inv))


#### **`STEP 8`** - Table Columns

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** We combine all the data collected above into a single combined dataframe for the chosen interaction type.

In [ ]:
##################################################
## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
##################################################
props_table = pd.DataFrame(props)
props_table.rename(columns={'label': 'idx'}, inplace=True)
props_table.drop(columns=['slice'], inplace=True)
props_table.insert(0, 'label',value=indexes[overlap_orgs])
props_table.insert(0, "object", overlap_orgs)
props_table.rename(columns={"area": "volume"}, inplace=True)
props_table.insert(11, "surface_area", surface_area_tab)
props_table.insert(13, "SA_to_volume_ratio", 
props_table["surface_area"].div(props_table["volume"]))
if scale is not None:
    round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
    props_table.insert(loc=2, column="scale", value=f"{round_scale}")
else: 
    props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(site.ndim))}")
props_table.insert((props_table.columns.get_loc('object')+1), f'{mask_name}_number', value=cells)
props_table.insert((props_table.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=regions)

pd.set_option('display.max_columns', None)
print("Interaction Metrics Table:")
display(props_table)

#### **`STEP 9`** - Run Distribution Metrics

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** Here, distribution metrics of the interactions are considered. This step is optional for the interaction metrics, and will not be performed if undesired. Check the method_distribution.ipynb jupyter notebook for more details.

In [ ]:
######################################################
## optional: DISTRIBUTION OF INTERACTION MEASUREMENTS
######################################################
if include_dist:
    interaction_dist_tab_list = []
    subregions = [mask_name] + list(regions_dict.keys())
    for region_name in subregions:
        centering_obj = dist_centering_obj[subregions.index(region_name)]
        if centering_obj != None:
            region_seg= regions[regions_file_names.index(region_name)]
            centering = regions[regions_file_names.index(centering_obj)]
            XY_interaction_dist, XY_bins, XY_wedges = get_XY_distribution(mask=mask, 
                                                                          mask_name=mask_name,
                                                                          obj=site,
                                                                          obj_name=overlap_orgs,
                                                                          region_seg=region_seg,
                                                                          region_name=region_name,
                                                                          centering_obj=centering,
                                                                          scale=scale,
                                                                          center_on=dist_center_on,
                                                                          keep_center_as_bin=dist_keep_center_as_bin,
                                                                          num_bins=dist_num_bins,
                                                                          zernike_degrees=dist_zernike_degrees)
                
            Z_interaction_dist = get_Z_distribution(mask=mask,
                                                        mask_name=mask_name,
                                                        obj=site,
                                                        region_seg=region_seg,
                                                        region_name=region_name,
                                                        obj_name=overlap_orgs,
                                                        center_obj=centering,
                                                        scale=scale)
            interaction_dist_tab_list.append(pd.merge(XY_interaction_dist, Z_interaction_dist, on=["object", "scale", f"{mask_name}_number", "subregion"]))

    interaction_dist_tab = pd.concat(interaction_dist_tab_list)
    indexes.clear()

print("Interaction Distribution Table:")
display(interaction_dist_tab)

#### **`STEP 10`** - Determine higher order overlaps

In [ ]:
LOc_NR = site.copy()                      
for org, val in organelle_segs.items():         
    if (org not in overlap_orgs.split(splitter)
        and np.any(site.astype(int)*val.astype(int))):            
        digit = len(str(np.max(val)))           
        valid = (LOc_NR>0)*(val>0)              
        HOc = (LOc_NR*(10**(digit)))+val        
        HOc[valid.astype(bool)==False]=0        
        HOc = label(HOc)                    
        for num, id in enumerate(np.unique(site[HOc > 0])):
            LOc_NR[LOc_NR==id] = 0   

#### **`STEP 11`** - Add column showing which overlaps are in a higher order

In [ ]:
LOc_NR = apply_mask((LOc_NR>0), mask).astype(int) * site
redundancy = props_table['idx'].isin(np.unique(LOc_NR[LOc_NR>0]).tolist())
props_table.insert((props_table.columns.get_loc('subregion')+1), "in_higher_order", list(map(bool, ~redundancy)))
props_table.drop(columns=['idx'], inplace=True)


print("Interaction Metrics Table With Novelty Specified:")
display(props_table)

#### **`DEFINE`** - create_overlap() function

The following code includes an example of how the method to create overlaps above is turned into one function. 

This function can utilized from infer-subc using:
```python
infer_subc.quantification.interactions.create_overlap()
```

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block defines the `create_overlap()` function. It is applied below.

In [ ]:
def _create_overlap(orgs:str,
                   organelle_segs: dict[str:np.ndarray],
                   splitter: str="X") -> tuple[np.ndarray, np.ndarray]: 
    ##########################################
    ## CREATE OVERLAP
    ##########################################
    site = np.ones_like(organelle_segs[orgs.split(splitter)[0]]) 
    for org in orgs.split(splitter):        
        b = organelle_segs[org]             
        valid = (b>0)*(site>0)              
        digit = len(str(np.max(site)))      
        site = (b*(10**(digit)))+site       
        site[valid.astype(bool)==False]=0   
        site = label(site)             
    return site

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test interaction. The settings specified above are applied here.

In [ ]:
overlap_seg = _create_overlap(overlap_orgs, organelle_segs, splitter=splitter)

print("The interaction segmentation here matches the segmentation created above:")
print(f"{overlap_seg.equals(site)}")

plugin_overlap_seg = create_overlap(overlap_orgs, organelle_segs, splitter=splitter)

print("The interaction segmentation here matches the segmentation created using the plugin:")
print(f"{overlap_seg.equals(plugin_overlap_seg)}")

#### **`DEFINE`** - interaction_metric_analysis() function

The following code includes an example of how the interactions steps above are combined into one function. This function can quantify the morphology of the interactions sites. It is applied in the batch process functions available in [_____]() notebooks. This function however, does not include the details of which interactions are present in higher order interactions or not.

This function can utilized from infer-subc using:
```python
infer_subc.quantification.interactions.interaction_metric_analysis()
```

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block defines the `interaction_metric_analysis()` function. It is applied below.

In [ ]:

def _interaction_metric_analysis(overlap_ID: str,
                                list_obj_names: list[str],
                                list_obj_segs: list[np.ndarray],
                                mask: np.ndarray,
                                mask_name: str,
                                regions_dict: dict[str:np.ndarray],
                                list_region_segs: Union[list[np.ndarray], None] = None,
                                list_region_names: Union[list[str], None] = None,
                                splitter: str="X",
                                scale: Union[tuple, None]=None,
                                include_dist:bool=False, 
                                dist_centering_obj: Union[np.ndarray, None]=None,
                                dist_num_bins: Union[int, None]=None,
                                dist_zernike_degrees: Union[int, None]=None,
                                dist_center_on: Union[bool, None]=None,
                                dist_keep_center_as_bin: Union[bool, None]=None,
                                return_site: bool=False):
    #########################
    ## CREATE ORG_DICT
    #########################
    org_dict = make_dict(list_obj_names, list_obj_segs)


    #########################
    ## CREATE OVERLAP REGIONS
    #########################
    # run create overlap function
    site = _create_overlap(overlap_ID, org_dict, splitter)

    #############################################################################################
    #assert the nth order overlap to within the cellmask
    labels = label(apply_mask(site, mask)).astype(int)


    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # start with LABEL
    properties = ["label"]

    # add position
    properties += ["centroid", "bbox"]

    # add area
    properties += ["area", "equivalent_diameter"] # "num_pixels", 

    # add shape measurements - NOTE: can't include minor axis measure because some of the contact sites are only one pixel
    properties += ["extent", "euler_number", "solidity", "axis_major_length", "slice"] # "feret_diameter_max",  , "axis_minor_length"
    

    ##################
    ## RUN REGIONPROPS
    ##################
    props = regionprops_table(labels, 
                              intensity_image=None, 
                              properties=properties, 
                              extra_properties=None, 
                              spacing=scale)

    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))

    #################################################################################################


    ########################################################
    ## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
    ########################################################
    over_inv = []
    involved = overlap_ID.split(splitter)
    indexes = {overlap_ID: []}

    cells = cell_finder(scale=scale, obj=labels, mask=mask, props=props)
    regions = region_finder(scale=scale, obj=labels, regions=regions_dict, props=props)

    for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = labels[props["slice"][index]]
            lorg = org_dict[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}")
            over_inv.append(f"{all_inv[0]}")
        indexes[overlap_ID].append('_'.join(over_inv))

        
    ##################################################
    ## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
    ##################################################
    props_table = pd.DataFrame(props)
    props_table.rename(columns={'label': 'idx'}, inplace=True)
    props_table.drop(columns=['slice'], inplace=True)
    props_table.insert(0, 'label',value=indexes[overlap_ID])
    props_table.insert(0, "object", overlap_ID)
    props_table.rename(columns={"area": "volume"}, inplace=True)
    props_table.insert(11, "surface_area", surface_area_tab)
    props_table.insert(13, "SA_to_volume_ratio", 
    props_table["surface_area"].div(props_table["volume"]))
    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}")
    props_table.insert((props_table.columns.get_loc('object')+1), f'{mask_name}_number', value=cells)
    props_table.insert((props_table.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=regions)
    
    ######################################################
    ## optional: DISTRIBUTION OF INTERACTION MEASUREMENTS
    ######################################################
    if include_dist:
        interaction_dist_tab_list = []
        subregions = [mask_name] + list(regions_dict.keys())
        for region_name in subregions:
            centering_obj = dist_centering_obj[subregions.index(region_name)]
            if centering_obj != None:
                region_seg= list_region_segs[list_region_names.index(region_name)]
                centering = list_region_segs[list_region_names.index(centering_obj)]


                # Separate fn for distribution combining Z and XY

                XY_interaction_dist, XY_bins, XY_wedges = get_XY_distribution(mask=mask, 
                                                                              mask_name=mask_name,
                                                                              obj=site,
                                                                              obj_name=overlap_ID,
                                                                              region_seg=region_seg,
                                                                              region_name=region_name,
                                                                              centering_obj=centering,
                                                                              scale=scale,
                                                                              center_on=dist_center_on,
                                                                              keep_center_as_bin=dist_keep_center_as_bin,
                                                                              num_bins=dist_num_bins,
                                                                              zernike_degrees=dist_zernike_degrees)
                
                Z_interaction_dist = get_Z_distribution(mask=mask,
                                                        mask_name=mask_name,
                                                        obj=site,
                                                        region_seg=region_seg,
                                                        region_name=region_name,
                                                        obj_name=overlap_ID,
                                                        center_obj=centering,
                                                        scale=scale)
                interaction_dist_tab_list.append(pd.merge(XY_interaction_dist, Z_interaction_dist, on=["object", "scale", f"{mask_name}_number", "subregion"]))

        interaction_dist_tab = pd.concat(interaction_dist_tab_list)
        indexes.clear()
        if return_site:
            return site, props_table, interaction_dist_tab
        else:
            return props_table, interaction_dist_tab
    else:
        indexes.clear()
        if return_site:
            return site, props_table 
        else:
            return props_table

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test interaction. The settings specified above are applied here.

In [ ]:
interaction_table = _interaction_metric_analysis(overlap_ID=overlap_orgs, 
                                                 list_obj_names=org_file_names,
                                                 list_obj_segs=organelle_segs_list,
                                                 mask=mask,
                                                 mask_name=mask_name,
                                                 regions_dict=regions_dict,
                                                 list_region_names=regions_file_names,
                                                 list_region_segs=regions,
                                                 splitter=splitter,
                                                 scale=scale,
                                                 include_dist=False,
                                                 dist_centering_obj=dist_centering_obj,
                                                 dist_num_bins=dist_num_bins,
                                                 dist_zernike_degrees=dist_zernike_degrees,
                                                 dist_center_on=dist_center_on,
                                                 dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                 return_site=False)

print("The interaction quantification here matches the quantification created above:")
print(f"{interaction_table.equals(props_table)}")

plugin_interaction_table = interaction_metric_analysis(overlap_ID=overlap_orgs, 
                                                       list_obj_names=org_file_names,
                                                       list_obj_segs=organelle_segs_list,
                                                       mask=mask,
                                                       mask_name=mask_name,
                                                       regions_dict=regions_dict,
                                                       list_region_names=regions_file_names,
                                                       list_region_segs=regions,
                                                       splitter=splitter,
                                                       scale=scale,
                                                       include_dist=False,
                                                       dist_centering_obj=dist_centering_obj,
                                                       dist_num_bins=dist_num_bins,
                                                       dist_zernike_degrees=dist_zernike_degrees,
                                                       dist_center_on=dist_center_on,
                                                       dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                       return_site=False)

print("The interaction quantification here matches the quantification created by the plugin:")
print(f"{interaction_table.equals(plugin_interaction_table)}")


#### **`DEFINE`** - find_novel_overlaps() function

The following code utilizes the both the overlap segmentation and the organelle segmentations to create a version of the overlap segmentation where all overlaps present in higher order overlaps are removed. This new overlap segmentation is then later used to compare with the original overlap segmentation data.

This function can utilized from infer-subc using:
```python
infer_subc.quantification.interactions.find_novel_overlaps()
```

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block defines the `find_novel_overlaps()` function. It is applied below.

In [ ]:
def _find_novel_overlaps(site: np.ndarray,
                        orgs: str,
                        organelle_segs: dict[str:np.ndarray],
                        splitter: str="X"):
    ##########################################
    ## DETERMINE NOVEL OVERLAPS
    ##########################################
    LOc_NR = site.copy()                      
    for org, val in organelle_segs.items():         
        if (org not in orgs.split(splitter)
            and np.any(site.astype(int)*val.astype(int))):            
            digit = len(str(np.max(val)))           
            valid = (LOc_NR>0)*(val>0)              
            HOc = (LOc_NR*(10**(digit)))+val        
            HOc[valid.astype(bool)==False]=0        
            HOc = label(HOc)                    
            for num, id in enumerate(np.unique(site[HOc > 0])):
                LOc_NR[LOc_NR==id] = 0    
    return LOc_NR

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block applies the function above to your test interaction. The settings specified above are applied here.

In [ ]:
LOi_NR = _find_novel_overlaps(site, overlap_orgs, organelle_segs, splitter)
LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site

print("The novel overlap segmentation here matches the novel overlap segmentation created above:")
print(f"{LOi_NR.equals(LOc_NR)}")

plugin_LOi_NR = find_novel_overlaps(site, overlap_orgs, organelle_segs, splitter)
plugin_LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site

print("The novel overlap segmentation here matches the novel overlap segmentation created above:")
print(f"{LOi_NR.equals(plugin_LOi_NR)}")

-----
### 🎉 **CONGRATULATIONS!! You've completed the `Interaction` method explanation notebook.**

This method is utilized in the following batch processing notebooks:
- [2.2_organelle_interactions](2.2_organelle_interactions.ipynb)

Continue on to learn about the other methods included in `infer-subc`:
- [method_distribution](method_distribution.ipynb)
- [method_morphology](method_morphology.ipynb)
